# avg igual ao da interpol - cenario 1

In [13]:
#com valores 60 e 120 hard coded

import random
import numpy as np
import math
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt  # Importar matplotlib para plotar gráficos
from functools import partial  # Adicionar no início do código
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

NUM_UAVS = 4
lambda_0 = 0.125
P_INTERFERENCE_DBM = 100
P_NOISE_DBM = -100
BANDWIDTH = 20e6
TRANSMIT_POWER_DBM = 20
MINDIST = 20
RESOLUTION = 20
AREA_SIZE = 100  # Tamanho da área em metros

# Definir os tipos de indivíduos e fitness
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximizar o fitnessAS
creator.create("Individual", list, fitness=creator.FitnessMax)  # Indivíduo é uma listaas

#ATENCAO JAMMER POSITION NO CREATE INDIVIDUAL

def create_individual(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, epoch_total_length=200, previous_best_solution=None, initial_positions=None, jammer_position=None):
    individual = []
    
    # Definir as posições iniciais de acordo com o epoch
    if epoch == 0:
        start_positions = initial_positions
    else:
        start_positions = []
        for uav in range(num_uavs):
            last_timeslot_idx = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            start_positions.append(previous_best_solution[last_timeslot_idx:last_timeslot_idx + 2])
    
    # Gerar posições finais CONTÍNUAS sem restrição de distância mínima
    final_positions = []
    
    for uav in range(num_uavs):
        final_x = random.uniform(60, 120)
        final_y = random.uniform(0, 60)
        final_positions.append((final_x, final_y))
    
    # Calcular todas as posições intermediárias
    for t in range(num_timeslots):
        for uav in range(num_uavs):
            start_x, start_y = start_positions[uav]
            end_x, end_y = final_positions[uav]
            
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            x = start_x + alpha * (end_x - start_x)
            y = start_y + alpha * (end_y - start_y)
            
            individual.extend([x, y])
    
    return creator.Individual(individual)

def print_communication_values_per_timeslot(individual, num_uavs, num_timeslots, jammer_position):
    all_min_capacities = []
    all_interference_matrices = []
    all_fitness_values = []  # ← ADICIONAR para armazenar fitness de cada timeslot

    for t in range(num_timeslots):
        angles = []
        positions = []
        
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        comm_matrix, interference_matrix = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)
        all_interference_matrices.append(interference_matrix)

        # ← USAR A MESMA LÓGICA DO evaluate_individual
        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))
                    except:
                        pass
        
        # Cálculo usando apenas links usados (IGUAL AO evaluate_individual)
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
            all_min_capacities.extend(capacidades_usadas)  # ← Adicionar todas as capacidades usadas
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Calcular fitness do timeslot (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        all_fitness_values.append(fitness)

    # Calcular fitness médio (IGUAL AO evaluate_individual)
    if len(all_fitness_values) > 0:
        avg_fitness = sum(all_fitness_values) / len(all_fitness_values)
    else:
        avg_fitness = 0.0

    if len(all_min_capacities) > 0:
        avg_min_capacity = sum(all_min_capacities) / len(all_min_capacities)
    else:
        avg_min_capacity = 0.0

    return all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness  # ← ADICIONAR avg_fitness

# Função para calcular a capacidade de comunicação entre todos os UAVs
def calculate_communication_capacity(antenna_angles, alignments, positions, jammer_position):
    communication_capacity = np.zeros((NUM_UAVS, NUM_UAVS))
    interference_matrix = np.full((NUM_UAVS, NUM_UAVS), P_INTERFERENCE_DBM)

    for i in range(NUM_UAVS):
        null_dir_i = antenna_angles[i]

        dx_jam = jammer_position[0] - positions[i][0]
        dy_jam = jammer_position[1] - positions[i][1]
        dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

        G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

        dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
        P_interf_dBm_i = interference_from_jammer(P_INTERFERENCE_DBM, G_jammer_i, lambda_0, dist_jammer_i)

        interference_matrix[i, :] = P_interf_dBm_i

    for i in range(NUM_UAVS):
        for j in range(NUM_UAVS):
            if i != j:
                null_dir_i = antenna_angles[i]
                null_dir_j = antenna_angles[j]

                dx_ij = positions[j][0] - positions[i][0]
                dy_ij = positions[j][1] - positions[i][1]
                dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                dx_ji = positions[i][0] - positions[j][0]
                dy_ji = positions[i][1] - positions[j][1]
                dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                capacity = calcular_capacidade_link(
                    pos1=positions[i],
                    pos2=positions[j],
                    lambda_0=lambda_0,
                    P_tx_dBm=TRANSMIT_POWER_DBM,
                    G_tx_dB=G_tx,
                    G_rx_dB=G_rx,
                    P_interference_dBm=interference_matrix[i, j],
                    P_noise_dBm=P_NOISE_DBM,
                    bandwidth=BANDWIDTH
                )

                communication_capacity[i][j] = capacity

    return communication_capacity, interference_matrix

def evaluate_individual(individual, num_uavs, num_timeslots, jammer_position):
    # Verificar se há pelo menos uma colisão
    
    # Se não há colisões, calcular o fitness normalmente
    alpha, beta = 1.0, 1.0
    total_fitness = 0

    for t in range(num_timeslots):
        positions = []
        angles = []
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)

        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Fitness sem penalização por colisões (já sabemos que não há colisões)
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        total_fitness += fitness

    # Calcular a média do fitness total
    average_fitness = total_fitness / num_timeslots if num_timeslots > 0 else 0
    return (average_fitness,)

def mutate_individual(individual, min_x, max_x, min_y, max_y, mutation_rate, num_uavs, num_timeslots, epoch, timeslot_length, epoch_total_length):
    
    for uav in range(num_uavs):
        if random.random() < mutation_rate:
            # Extrair posições iniciais
            start_x = individual[uav * 2]
            start_y = individual[uav * 2 + 1]
            
            # Gerar nova posição final aleatória sem restrição de distância mínima
            new_final_x = random.uniform(60, 120)
            new_final_y = random.uniform(0, 60)
            
            # Atualizar a posição final
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2] = new_final_x
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1] = new_final_y
            
            # Recalcular posições intermediárias
            for t in range(1, num_timeslots):
                alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                x = start_x + alpha * (new_final_x - start_x)
                y = start_y + alpha * (new_final_y - start_y)
                
                idx_x = t * num_uavs * 2 + uav * 2
                idx_y = t * num_uavs * 2 + uav * 2 + 1
                individual[idx_x] = x
                individual[idx_y] = y
    
    return individual,

def custom_crossover(ind1, ind2, num_uavs):
    num_timeslots = len(ind1) // (num_uavs * 2)
    
    # Trocar posições finais entre os indivíduos sem restrição de distância mínima
    for uav in range(num_uavs):
        if random.random() < 0.8:
            # Índices das posições finais (último timeslot)
            final_idx_x = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            final_idx_y = (num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1
            
            # Trocar as posições finais
            ind1[final_idx_x], ind2[final_idx_x] = ind2[final_idx_x], ind1[final_idx_x]
            ind1[final_idx_y], ind2[final_idx_y] = ind2[final_idx_y], ind1[final_idx_y]
            
            # Recalcular posições intermediárias para ambos os indivíduos
            for ind in [ind1, ind2]:
                start_x = ind[uav * 2]
                start_y = ind[uav * 2 + 1]
                end_x = ind[final_idx_x]
                end_y = ind[final_idx_y]
                
                for t in range(1, num_timeslots - 1):
                    alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                    idx_x = t * num_uavs * 2 + uav * 2
                    idx_y = t * num_uavs * 2 + uav * 2 + 1
                    ind[idx_x] = start_x + alpha * (end_x - start_x)
                    ind[idx_y] = start_y + alpha * (end_y - start_y)
    
    return ind1, ind2

# Configuração do DEAP atualizada
def setup_deap(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, crossover_rate, mutation_rate, initial_positions, previous_best_solution=None, jammer_position=None, epoch_total_length=300):
    toolbox = base.Toolbox()
    
    # Registrar funções
    toolbox.register("individual", create_individual, 
                     num_uavs=num_uavs, 
                     num_timeslots=num_timeslots, 
                     timeslot_length=timeslot_length, 
                     epoch=epoch, 
                     min_y=min_y, 
                     max_y=max_y,
                     epoch_total_length=epoch_total_length,  # ← ADICIONAR
                     previous_best_solution=previous_best_solution,
                     initial_positions=initial_positions,
                     jammer_position=jammer_position)  # Passar a posição do jammer aqui
    
    # O restante do código permanece o mesmo...

    
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate_individual, num_uavs=num_uavs, num_timeslots=num_timeslots, jammer_position=jammer_position)
    
    # Usar partial para fixar o argumento num_uavs na função custom_crossover
    toolbox.register("mate", partial(custom_crossover, num_uavs=num_uavs))
    
    min_x = epoch * num_timeslots * timeslot_length
    max_x = (epoch * num_timeslots + num_timeslots) * timeslot_length
    
    toolbox.register("mutate", mutate_individual, 
                 min_x=min_x, 
                 max_x=max_x, 
                 min_y=min_y, 
                 max_y=max_y, 
                 mutation_rate=mutation_rate,
                 num_uavs=num_uavs,
                 num_timeslots=num_timeslots,
                 epoch=epoch,
                 timeslot_length=timeslot_length,
                 epoch_total_length=epoch_total_length)  # ← ADICIONAR
    
    toolbox.register("select", tools.selTournament, tournsize=3)  # Seleção por torneio
    
    return toolbox

def genetic_algorithm(num_uavs, num_timeslots, timeslot_length, population_size, generations, crossover_rate, mutation_rate, epoch, initial_positions, previous_best_solution=None, jammer_position=None, min_y=0.0, max_y=5.0, epoch_total_length=300):
    # Configurar o DEAP
    toolbox = setup_deap(
        num_uavs=num_uavs,
        num_timeslots=num_timeslots,
        timeslot_length=timeslot_length,
        epoch=epoch,
        min_y=min_y,
        max_y=max_y,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        initial_positions=initial_positions,
        previous_best_solution=previous_best_solution,
        jammer_position=jammer_position,
        epoch_total_length=epoch_total_length  # ← ADICIONAR
    )

    
    # Criar população inicial
    population = toolbox.population(n=population_size)
    
    # Avaliar a população inicial
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
    
    # Configurar estatísticas para impressão
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Listas para armazenar os dados de cada geração
    gen_list = []
    avg_list = []
    std_list = []
    min_list = []
    max_list = []
    
    # Executar o algoritmo genético com eaSimple
    for gen in range(generations):
        # Avançar uma geração
        algorithms.eaSimple(
            population, 
            toolbox, 
            cxpb=crossover_rate,  # Probabilidade de cruzamento
            mutpb=mutation_rate,  # Probabilidade de mutação
            ngen=1,               # Apenas uma geração por iteração
            stats=stats,          # Estatísticas para impressão
            verbose=False         # Desativar impressão da tabela para cada geração
        )
        
        # Coletar os dados da geração atual
        record = stats.compile(population)
        gen_list.append(gen)
        avg_list.append(record["avg"])
        std_list.append(record["std"])
        min_list.append(record["min"])
        max_list.append(record["max"])
        
        # Escrever os valores no arquivo
        #write_fitness_values(epoch, gen, record["avg"], record["max"], record["min"], record["std"])
    
    # Retornar o melhor indivíduo
    best_individual = tools.selBest(population, k=1)[0]
    return best_individual

# Função para gerar posições iniciais dos UAVs
def generate_initial_positions(num_uavs, min_y, max_y, timeslot_length, manual=False, manual_positions=None):
    if manual and manual_positions is not None:
        return manual_positions  # Usa as posições fornecidas

    positions = []
    max_attempts = 1000  # para evitar loops infinitos

    for _ in range(num_uavs):
        attempts = 0
        while True:
            y = random.uniform(min_y, max_y)
            x = random.uniform(0, timeslot_length)  # entre -timeslot_length e 0
            
            candidate = (x, y)
            
            # Verifica se está longe o suficiente das outras posições já geradas
            if all(np.linalg.norm(np.array(candidate) - np.array(pos)) >= MINDIST for pos in positions):
                positions.append(candidate)
                break
            
            attempts += 1
            if attempts >= max_attempts:
                # Se não conseguir, aceita a posição mesmo assim para evitar bloqueio
                positions.append(candidate)
                break
                
    return positions

# def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
#     with open(filename, "a") as file:
#         # Escrever apenas a melhor solução (que já inclui tudo)
#         individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
#         file.write(f"{individual_str}\n \n")

def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            # Processar o conteúdo existente se necessário
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        # Se o arquivo não existir, não há conteúdo para ler
        print("O arquivo não existe. Criando um novo arquivo.")

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")


def write_fitness_values(epoch, gen, avg, max_val, min_val, std, filename="fitness_values.txt"):
    
    with open(filename, "a") as file:
        if gen == 0:  # Escrever o cabeçalho no início de cada epoch
            file.write(f"=== Epoch {epoch + 1} ===\n")
            file.write("gen\tavg\tmax\tmin\tstd\n")
        file.write(f"{gen}\t{avg:.4f}\t{max_val:.4f}\t{min_val:.4f}\t{std:.4f}\n")

# Função principal atualizada
def simulate_uavs_with_ga(num_epochs, num_timeslots, timeslot_length, num_uavs, population_size=50, generations=50, crossover_rate=0.9, mutation_rate=0.3, manual_initial_positions=None, jammer_position=None, min_y=0.0, max_y=10.0, epoch_total_length=300):
    # Gerar posições iniciais dos UAVs
    initial_positions = generate_initial_positions(
        num_uavs, min_y, max_y, timeslot_length,
        manual=manual_initial_positions is not None,
        manual_positions=manual_initial_positions
    )

    previous_best_solution = None
    
    for epoch in range(num_epochs):
        # Executar o algoritmo genético
        best_solution = genetic_algorithm(
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            timeslot_length=timeslot_length,
            population_size=population_size,
            generations=generations,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            epoch=epoch,
            initial_positions=initial_positions,
            previous_best_solution=previous_best_solution,
            jammer_position=jammer_position,  # Passar a posição do jammer aqui
            min_y=min_y,
            max_y=max_y,
            epoch_total_length=epoch_total_length  # ← ADICIONAR
        )

        # O restante do código permanece o mesmo...


        # Salvar a melhor solução em um arquivo de texto
        save_best_solution_to_file(best_solution, initial_positions)
        
        all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness = print_communication_values_per_timeslot(best_solution, num_uavs, num_timeslots, jammer_position)

        # Guardar a melhor solução para a próxima epoch


        # # Calcular e imprimir a coerência dos valores
        # best_fitness = best_solution.fitness.values[0]
        # recalculated_fitness = evaluate_individual(best_solution, num_uavs, num_timeslots, jammer_position)[0]
        # # Ler o último valor máximo do fitness_values.txt
        # with open("fitness_values.txt", "r") as file:
        #     lines = file.readlines()
        #     last_max_fitness = float(lines[-1].split("\t")[2])  # Último max na última linha
        # # Comparar os valores
        # print(f"Fitness da melhor solução: {best_fitness:.4f}")
        # print(f"Fitness recalculado: {recalculated_fitness:.4f}")
        # print(f"Último max no arquivo: {last_max_fitness:.4f}")
        # print(f"São iguais? {'✅' if abs(best_fitness - last_max_fitness) < 1 else '❌'}")
        # # Guardar a melhor solução para a próxima epoch
        # previous_best_solution = best_solution



        previous_best_solution = best_solution

        return best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity


d:\Desktop\repo\gym\gym_GA_env\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
d:\Desktop\repo\gym\gym_GA_env\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [14]:
# Posições que respeitam MINDIST = 5 metros
posicoes_manuais1 = [
    (20,60),   # UAV 110.172259033584051,85.75523376648344,75.88992564121607,16.084203954794695,31.066957304533616,49.76376031917511,69.18455395110837,76.3309773445626,
    (20,40),   # UAV 2 (60m de distância do UAV1)
    (20,0),   # UAV 3 (60m de distância do UAV1)
    (20,20)    # UAV 4 (60m de distância dos outros)
]

# Executar simulação
best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity = simulate_uavs_with_ga(
    num_epochs=1, 
    num_timeslots=6, 
    timeslot_length=60, 
    num_uavs=NUM_UAVS,
    population_size=1,
    generations=1,
    manual_initial_positions=posicoes_manuais1,
    jammer_position=np.array([0,500]),
    min_y=0.0,
    max_y=60.0,
    epoch_total_length=120
)

print("Posições iniciais usadas:", initial_positions)
print("Melhor solução encontrada:", best_solution)
print("Posição do jammer:", jammer_position)

O arquivo não existe. Criando um novo arquivo.
Posições iniciais usadas: [(20, 60), (20, 40), (20, 0), (20, 20)]
Melhor solução encontrada: [20.0, 60.0, 20.0, 40.0, 20.0, 0.0, 20.0, 20.0, 35.975201452018425, 50.57354425846998, 32.97938854193155, 40.85895093061311, 32.54448823811742, 10.043765053121364, 36.22194661480026, 24.319463798173125, 51.95040290403685, 41.14708851693996, 45.9587770838631, 41.717901861226224, 45.08897647623483, 20.087530106242728, 52.44389322960052, 28.63892759634625, 67.92560435605527, 31.72063277540995, 58.938165625794646, 42.576852791839336, 57.63346471435225, 30.13129515936409, 68.66583984440078, 32.958391394519374, 83.9008058080737, 22.294177033879926, 71.9175541677262, 43.43580372245245, 70.17795295246967, 40.175060212485455, 84.88778645920104, 37.2778551926925, 99.87600726009212, 12.867721292349913, 84.89694270965775, 44.29475465306556, 82.72244119058709, 50.21882526560682, 101.1097330740013, 41.597318990865624]
Posição do jammer: [  0 500]


# analisar

In [15]:
def analisar_comunicacoes_detalhado(filename, num_uavs, num_timeslots, jammer_position):
    def parse_solution_file(filename):
        with open(filename, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]
        
        solutions = []
        current_solution = []
        for line in lines:
            if line.startswith('==='):  # Nova época
                if current_solution:
                    solutions.append(current_solution)
                    current_solution = []
            else:
                current_solution.extend(map(float, line.split()))
        if current_solution:
            solutions.append(current_solution)
        return solutions

    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # Processar arquivo de saída
    solucoes = parse_solution_file(filename)
    
    resultados = []
    for idx, solucao in enumerate(solucoes):
        print(f"\nAnálise para Época {idx+1}:")
        
        # 🚨 VERIFICAR COLISÕES PRIMEIRO (IGUAL AO evaluate_individual)
        #tem_colisoes = has_collision(solucao, num_uavs, num_timeslots)
        #print(f"🔍 Verificação de colisões: {'❌ TEM COLISÕES' if tem_colisoes else '✅ SEM COLISÕES'}")
        
        # if tem_colisoes:
        #     print(f"🚨 SOLUÇÃO COM COLISÕES DETECTADA!")
        #     print(f"   Fitness = 0.0 (igual ao evaluate_individual)")
        #     print(f"   Não será feita análise detalhada.")
        #     print("="*50)
        #     return [(0.0, "Solução com colisões")]
        
        # Se não há colisões, continuar com a análise normal
        print(f"✅ Solução válida - prosseguindo com análise detalhada...")
        
        # Lista para armazenar todos os C_média_total e C_min_total do epoch
        todos_c_media = []
        todos_c_min = []
        
        # Recriar a solução para cada timeslot
        for t in range(num_timeslots):
            print(f"\nTimeslot {t+1}:")
            positions = []
            angles = []
            
            for i in range(num_uavs):
                idx_pos = t * num_uavs * 2 + i * 2  # MUDANÇA: *2, pois só temos x e y
                x, y = solucao[idx_pos:idx_pos+2]
                positions.append(np.array([x, y]))
                
                # Calcular ângulo dinamicamente
                angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
                angles.append(angle)
            
            # Calcular matriz de comunicação
            comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
            
            # Gerar métricas detalhadas
            metricas = avaliar_grafo(comm_matrix)
            
            # Exibir resultados
            print("\nMatriz de Comunicação (bps):")
            print(np.round(comm_matrix, 2))
            
            print("\nResumo de Capacidades:")
            print(f"- Capacidades: {metricas['capacidades']}")
            print(f"- Média: {np.mean(metricas['capacidades']):.2f} bps")
            print(f"- Mínima: {min(metricas['capacidades']):.2f} bps")
            print(f"- Máxima: {max(metricas['capacidades']):.2f} bps")
            
            print("\nCaminhos Críticos:")
            for (i,j), data in metricas['caminhos_minimos'].items():
                print(f"UAV {i} → UAV {j}: {data['path']} (Capacidade: {data['capacidade']:.2f} bps)")
            
            print("\nBottlenecks por UAV:")
            for uav, cap in metricas['bottlenecks'].items():
                print(f"UAV {uav}: {cap:.2f} bps")
            
            print(f"\nGrafo é fortemente conexo? {'Sim' if metricas['conectividade'] else 'Não'}")
            
            print("\nLinks Usados:")
            for u, v in metricas['links_usados']:
                print(f"{u} ↔ {v}")
            
            print("\nLinks Não Usados:")
            for u, v in metricas['links_nao_usados']:
                print(f"{u} ↔ {v}")
            
            # Cálculo do fitness usando apenas links usados
            capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
            if capacidades_usadas:
                C_media_total = np.mean(capacidades_usadas)
                C_min_total = min(capacidades_usadas)
            else:
                C_media_total = 0
                C_min_total = 0
            
            # Adicionar às listas do epoch
            todos_c_media.append(C_media_total)
            todos_c_min.append(C_min_total)
            
            print(f"\nCálculo do Fitness para Timeslot {t+1}:")
            print(f"Capacidades Usadas: {capacidades_usadas}")
            print(f"C_média_total = {C_media_total:.2f} bps")
            print(f"C_min_total = {C_min_total:.2f} bps")
            
            resultados.append({
                'epoch': idx+1,
                'timeslot': t+1,
                'metricas': metricas,
                'C_media_total': C_media_total,
                'C_min_total': C_min_total
            })
        
        # ✅ CORREÇÃO: Calcular fitness de cada timeslot e depois fazer a média (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness_por_timeslot = []
        
        for i in range(len(todos_c_media)):
            C_media = todos_c_media[i]
            C_min = todos_c_min[i]
            fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
            fitness_por_timeslot.append(fitness_timeslot)
        
        # Fitness médio do epoch (IGUAL AO evaluate_individual)
        fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
        
        # Calcular também as médias para informação
        media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
        media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
        
        print(f"\n" + "="*50)
        print(f"RESUMO DO EPOCH {idx+1}:")
        print(f"Todos os C_média_total: {[f'{x:.2f}' for x in todos_c_media]}")
        print(f"Todos os C_min_total: {[f'{x:.2f}' for x in todos_c_min]}")
        print(f"Fitness por timeslot: {[f'{x:.4f}' for x in fitness_por_timeslot]}")
        print(f"Média dos C_média_total: {media_c_media_epoch:.2f} bps")
        print(f"Média dos C_min_total: {media_c_min_epoch:.2f} bps")
        print(f"🎯 Fitness do Epoch (CORRETO) = {fitness_epoch:.4f}")
        print(f"✅ Agora deve bater com o valor do gráfico!")
        print("="*50)
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch


In [26]:
resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch = analisar_comunicacoes_detalhado(
    filename="best_solution_epoch.txt",
    num_uavs=4,  # ou o número correto de UAVs que você está usando
    num_timeslots=6,  # Confirmando que são 6 timeslots
    jammer_position=np.array([120.0,500.0])  # Posição do jammer
)

print(media_c_media_epoch)
print(media_c_min_epoch)




Análise para Época 1:
✅ Solução válida - prosseguindo com análise detalhada...

Timeslot 1:

Matriz de Comunicação (bps):
[[    0.     960.8    987.03   218.12]
 [  960.8      0.    7055.88 13334.  ]
 [  987.03  7055.88     0.    5700.91]
 [  218.12 13334.    5700.91     0.  ]]

Resumo de Capacidades:
- Capacidades: [960.7959575259172, 987.0326181386297, 218.12309232547778, 960.7959575259172, 7055.878879945144, 13334.002252154702, 987.0326181386297, 7055.878879945144, 5700.909677584164, 218.12309232547778, 13334.002252154702, 5700.909677584164]
- Média: 4709.46 bps
- Mínima: 218.12 bps
- Máxima: 13334.00 bps

Caminhos Críticos:
UAV 0 → UAV 1: [0, 1] (Capacidade: 960.80 bps)
UAV 0 → UAV 2: [0, 2] (Capacidade: 987.03 bps)
UAV 0 → UAV 3: [0, 1, 3] (Capacidade: 960.80 bps)
UAV 1 → UAV 0: [1, 0] (Capacidade: 960.80 bps)
UAV 1 → UAV 2: [1, 2] (Capacidade: 7055.88 bps)
UAV 1 → UAV 3: [1, 3] (Capacidade: 13334.00 bps)
UAV 2 → UAV 0: [2, 0] (Capacidade: 987.03 bps)
UAV 2 → UAV 1: [2, 1] (Capac

# dataset permutado para treianr modelo (já vem de trás) - uav_dataset_com_permutacoes.csv

In [ ]:
# já vem de trás igual ao class 1 DISCRETO

#1 mas é o DISCRETO

# criar dataset.csv (valores CONTINUOS) para testar as previsões da interpolação

In [ ]:
# igual ao interpol so copiar de lá - dataset_combined.csv e depois copiar tambem o com as permutaçoes

# XGBoost

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import os
import joblib
import warnings

warnings.filterwarnings('ignore')

class UAVPositionRegressor:
    def __init__(self, train_file, test_file):
        self.train_file = train_file
        self.test_file = test_file
        self.train_data = None
        self.test_data = None
        self.X_train = None
        self.y_train = None
        self.X_test = None
        self.y_test = None
        self.scaler = StandardScaler()
        self.model = None
        
    def load_data(self):
        """Carrega dados do CSV de treino e teste"""
        print("📂 Carregando dados de treino...")
        self.train_data = pd.read_csv(self.train_file)
        print(f"✅ {len(self.train_data)} linhas de treino carregadas")
        
        print("📂 Carregando dados de teste...")
        self.test_data = pd.read_csv(self.test_file)
        print(f"✅ {len(self.test_data)} linhas de teste carregadas")
        
        feature_cols = ['Jammer_X', 'Jammer_Y', 'Initial_X1', 'Initial_Y1', 
                        'Initial_X2', 'Initial_Y2', 'Initial_X3', 'Initial_Y3', 
                        'Initial_X4', 'Initial_Y4']
        
        target_cols = ['Final_X1', 'Final_Y1', 'Final_X2', 'Final_Y2',
                       'Final_X3', 'Final_Y3', 'Final_X4', 'Final_Y4']
        
        # Converter para numérico
        for col in feature_cols + target_cols:
            self.train_data[col] = pd.to_numeric(self.train_data[col], errors='coerce')
            self.test_data[col] = pd.to_numeric(self.test_data[col], errors='coerce')
        
        self.train_data = self.train_data.dropna(subset=feature_cols + target_cols)
        self.test_data = self.test_data.dropna(subset=feature_cols + target_cols)
        
        self.X_train = self.train_data[feature_cols].values
        self.y_train = self.train_data[target_cols].values
        
        self.X_test = self.test_data[feature_cols].values
        self.y_test = self.test_data[target_cols].values
        
        print(f"📊 Features de treino: {self.X_train.shape}")
        print(f"🎯 Valores de treino (8 coordenadas): {self.y_train.shape}")
        print(f"📊 Features de teste: {self.X_test.shape}")
        print(f"🎯 Valores de teste (8 coordenadas): {self.y_test.shape}")
        
    def split_and_scale(self):
        """Normaliza os dados de treino"""
        print("🔄 Normalizando dados de treino...")
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        
        print(f"📊 Dados de treino normalizados: {self.X_train_scaled.shape}")
        
    def train_model(self):
        """Treina o modelo XGBoost"""
        print("🚀 Treinando modelo XGBoost...")
        
        self.model = xgb.XGBRegressor(random_state=42, verbosity=0)
        
        # Treinar o modelo
        print("⏳ Treinando... (pode demorar com 300k+ amostras)")
        self.model.fit(self.X_train_scaled, self.y_train)
        
        # Criar pasta para salvar o modelo
        model_folder = 'reg_pkl'
        if not os.path.exists(model_folder):
            os.makedirs(model_folder)
            print(f"📁 Pasta '{model_folder}' criada")
        
        # Salvar o modelo
        joblib.dump(self.model, os.path.join(model_folder, 'xgboost_regressor_model.pkl'))
        joblib.dump(self.scaler, os.path.join(model_folder, 'xgboost_scaler.pkl'))  # Salvar scaler também
        print(f"✅ Modelo salvo como {model_folder}/xgboost_regressor_model.pkl")
        print(f"✅ Scaler salvo como {model_folder}/scaler.pkl")
        
    def evaluate_and_save_predictions(self):
        """Faz previsões no dataset de teste e salva em CSV"""
        print("\n📊 Fazendo previsões no dataset de teste...")
        
        # Normalizar os dados de teste
        X_test_scaled = self.scaler.transform(self.X_test)
        
        # Fazer previsões (retorna valores contínuos)
        y_pred = self.model.predict(X_test_scaled)
        
        print(f"🔍 Shape das previsões: {y_pred.shape}")
        print(f"🔍 Exemplo de previsão (valores): {y_pred[0]}")
        
        # Criar DataFrame para salvar resultados
        results_df = pd.DataFrame(
            np.hstack((self.X_test, y_pred, self.y_test)),
            columns=['Jammer_X', 'Jammer_Y', 'Initial_X1', 'Initial_Y1', 
                     'Initial_X2', 'Initial_Y2', 'Initial_X3', 'Initial_Y3', 
                     'Initial_X4', 'Initial_Y4', 
                     'Predicted_Final_X1', 'Predicted_Final_Y1', 
                     'Predicted_Final_X2', 'Predicted_Final_Y2', 
                     'Predicted_Final_X3', 'Predicted_Final_Y3', 
                     'Predicted_Final_X4', 'Predicted_Final_Y4', 
                     'Real_Final_X1', 'Real_Final_Y1', 
                     'Real_Final_X2', 'Real_Final_Y2', 
                     'Real_Final_X3', 'Real_Final_Y3', 
                     'Real_Final_X4', 'Real_Final_Y4']
        )
        
        # Salvar previsões em CSV
        results_folder = 'reg_results'
        if not os.path.exists(results_folder):
            os.makedirs(results_folder)
            print(f"📁 Pasta '{results_folder}' criada")
        
        results_df.to_csv(os.path.join(results_folder, 'predictions_xgboost_regressor.csv'), index=False)
        print(f"✅ Previsões salvas em {results_folder}/predictions_xgboost_regressor.csv")
        
        # 📊 CALCULAR MÉTRICAS DE REGRESSÃO
        print("\n📈 MÉTRICAS DE REGRESSÃO:")
        for i in range(self.y_test.shape[1]):
            mae = mean_absolute_error(self.y_test[:, i], y_pred[:, i])
            mse = mean_squared_error(self.y_test[:, i], y_pred[:, i])
            r2 = r2_score(self.y_test[:, i], y_pred[:, i])
            print(f"  Coordenada {i+1}: MAE = {mae:.4f}, MSE = {mse:.4f}, R² = {r2:.4f}")

# ================================
# USO
# ================================

# Treinar e prever
regressor = UAVPositionRegressor('uav_dataset_reference_3_com_permutacoes.csv', 'dataset_combined_com_permutacoes.csv')
regressor.load_data()
regressor.split_and_scale()
regressor.train_model()
regressor.evaluate_and_save_predictions()

print("\n🎉 Treino e previsões concluídos!")
print("📁 Modelo salvo em: reg_pkl/")
print("📊 Resultados salvos em: reg_results/")


📂 Carregando dados de treino...
✅ 305760 linhas de treino carregadas
📂 Carregando dados de teste...
✅ 2400 linhas de teste carregadas
📊 Features de treino: (305760, 10)
🎯 Valores de treino (8 coordenadas): (305760, 8)
📊 Features de teste: (2400, 10)
🎯 Valores de teste (8 coordenadas): (2400, 8)
🔄 Normalizando dados de treino...
📊 Dados de treino normalizados: (305760, 10)
🚀 Treinando modelo XGBoost...
⏳ Treinando... (pode demorar com 300k+ amostras)
✅ Modelo salvo como reg_pkl/xgboost_regressor_model.pkl
✅ Scaler salvo como reg_pkl/scaler.pkl

📊 Fazendo previsões no dataset de teste...
🔍 Shape das previsões: (2400, 8)
🔍 Exemplo de previsão (valores): [92.35546  33.218197 88.42579  30.578543 87.75681  30.372627 91.18351
 27.436022]
✅ Previsões salvas em reg_results/predictions_xgboost_regressor.csv

📈 MÉTRICAS DE REGRESSÃO:
  Coordenada 1: MAE = 14.1795, MSE = 279.9428, R² = -0.1045
  Coordenada 2: MAE = 10.1214, MSE = 156.1546, R² = 0.0755
  Coordenada 3: MAE = 14.1795, MSE = 279.9428,

# RF

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import os
import joblib
import warnings

warnings.filterwarnings('ignore')

class UAVPositionRegressor:
    def __init__(self, train_file, test_file):
        self.train_file = train_file
        self.test_file = test_file
        self.train_data = None
        self.test_data = None
        self.X_train = None
        self.y_train = None
        self.X_test = None
        self.y_test = None
        self.scaler = StandardScaler()
        self.model = None
        
    def load_data(self):
        """Carrega dados do CSV de treino e teste"""
        print("📂 Carregando dados de treino...")
        self.train_data = pd.read_csv(self.train_file)
        print(f"✅ {len(self.train_data)} linhas de treino carregadas")
        
        print("📂 Carregando dados de teste...")
        self.test_data = pd.read_csv(self.test_file)
        print(f"✅ {len(self.test_data)} linhas de teste carregadas")
        
        feature_cols = ['Jammer_X', 'Jammer_Y', 'Initial_X1', 'Initial_Y1', 
                        'Initial_X2', 'Initial_Y2', 'Initial_X3', 'Initial_Y3', 
                        'Initial_X4', 'Initial_Y4']
        
        target_cols = ['Final_X1', 'Final_Y1', 'Final_X2', 'Final_Y2',
                       'Final_X3', 'Final_Y3', 'Final_X4', 'Final_Y4']
        
        # Converter para numérico
        for col in feature_cols + target_cols:
            self.train_data[col] = pd.to_numeric(self.train_data[col], errors='coerce')
            self.test_data[col] = pd.to_numeric(self.test_data[col], errors='coerce')
        
        self.train_data = self.train_data.dropna(subset=feature_cols + target_cols)
        self.test_data = self.test_data.dropna(subset=feature_cols + target_cols)
        
        self.X_train = self.train_data[feature_cols].values
        self.y_train = self.train_data[target_cols].values
        
        self.X_test = self.test_data[feature_cols].values
        self.y_test = self.test_data[target_cols].values
        
        print(f"📊 Features de treino: {self.X_train.shape}")
        print(f"🎯 Valores de treino (8 coordenadas): {self.y_train.shape}")
        print(f"📊 Features de teste: {self.X_test.shape}")
        print(f"🎯 Valores de teste (8 coordenadas): {self.y_test.shape}")
        
    def split_and_scale(self):
        """Normaliza os dados de treino"""
        print("🔄 Normalizando dados de treino...")
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        
        print(f"📊 Dados de treino normalizados: {self.X_train_scaled.shape}")
        
    def train_model(self):
        """Treina o modelo Random Forest"""
        print("🚀 Treinando modelo Random Forest...")
        
        self.model = RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1  # Usa todos os núcleos disponíveis
        )
        
        print("⏳ Treinando... (pode demorar com 300k+ amostras)")
        self.model.fit(self.X_train_scaled, self.y_train)
        
        # Criar pasta para salvar o modelo
        model_folder = 'reg_pkl_rf'
        if not os.path.exists(model_folder):
            os.makedirs(model_folder)
            print(f"📁 Pasta '{model_folder}' criada")
        
        # Salvar o modelo e o scaler
        joblib.dump(self.model, os.path.join(model_folder, 'random_forest_regressor_model.pkl'))
        joblib.dump(self.scaler, os.path.join(model_folder, 'random_forest_scaler.pkl'))
        print(f"✅ Modelo salvo como {model_folder}/random_forest_regressor_model.pkl")
        print(f"✅ Scaler salvo como {model_folder}/random_forest_scaler.pkl")
  
    def evaluate_and_save_predictions(self):
        """Faz previsões no dataset de teste e salva em CSV"""
        print("\n📊 Fazendo previsões no dataset de teste...")
        
        # Normalizar os dados de teste
        X_test_scaled = self.scaler.transform(self.X_test)
        
        # Fazer previsões (retorna valores contínuos)
        y_pred = self.model.predict(X_test_scaled)
        
        print(f"🔍 Shape das previsões: {y_pred.shape}")
        print(f"🔍 Exemplo de previsão (valores): {y_pred[0]}")
        
        # Criar DataFrame para salvar resultados
        results_df = pd.DataFrame(
            np.hstack((self.X_test, y_pred, self.y_test)),
            columns=['Jammer_X', 'Jammer_Y', 'Initial_X1', 'Initial_Y1', 
                     'Initial_X2', 'Initial_Y2', 'Initial_X3', 'Initial_Y3', 
                     'Initial_X4', 'Initial_Y4', 
                     'Predicted_Final_X1', 'Predicted_Final_Y1', 
                     'Predicted_Final_X2', 'Predicted_Final_Y2', 
                     'Predicted_Final_X3', 'Predicted_Final_Y3', 
                     'Predicted_Final_X4', 'Predicted_Final_Y4', 
                     'Real_Final_X1', 'Real_Final_Y1', 
                     'Real_Final_X2', 'Real_Final_Y2', 
                     'Real_Final_X3', 'Real_Final_Y3', 
                     'Real_Final_X4', 'Real_Final_Y4']
        )
        
        # Salvar previsões em CSV
        results_folder = 'reg_results'
        if not os.path.exists(results_folder):
            os.makedirs(results_folder)
            print(f"📁 Pasta '{results_folder}' criada")
        
        results_df.to_csv(os.path.join(results_folder, 'predictions_rf_regressor.csv'), index=False)
        print(f"✅ Previsões salvas em {results_folder}/predictions_rf_regressor.csv")
        
        # 📊 CALCULAR MÉTRICAS DE REGRESSÃO
        print("\n📈 MÉTRICAS DE REGRESSÃO:")
        for i in range(self.y_test.shape[1]):
            mae = mean_absolute_error(self.y_test[:, i], y_pred[:, i])
            mse = mean_squared_error(self.y_test[:, i], y_pred[:, i])
            r2 = r2_score(self.y_test[:, i], y_pred[:, i])
            print(f"  Coordenada {i+1}: MAE = {mae:.4f}, MSE = {mse:.4f}, R² = {r2:.4f}")

# ================================
# USO
# ================================

# Treinar e prever
regressor = UAVPositionRegressor('uav_dataset_reference_3_com_permutacoes.csv', 'dataset_combined_com_permutacoes.csv')
regressor.load_data()
regressor.split_and_scale()
regressor.train_model()
regressor.evaluate_and_save_predictions()

print("\n🎉 Treino e previsões concluídos!")
print("📁 Modelo salvo em: reg_pkl/")
print("📊 Resultados salvos em: reg_results/")


📂 Carregando dados de treino...
✅ 305760 linhas de treino carregadas
📂 Carregando dados de teste...
✅ 2400 linhas de teste carregadas
📊 Features de treino: (305760, 10)
🎯 Valores de treino (8 coordenadas): (305760, 8)
📊 Features de teste: (2400, 10)
🎯 Valores de teste (8 coordenadas): (2400, 8)
🔄 Normalizando dados de treino...
📊 Dados de treino normalizados: (305760, 10)
🚀 Treinando modelo Random Forest...
⏳ Treinando... (pode demorar com 300k+ amostras)
📁 Pasta 'reg_pkl_rf' criada
✅ Modelo salvo como reg_pkl_rf/random_forest_regressor_model.pkl
✅ Scaler salvo como reg_pkl_rf/random_forest_scaler.pkl

📊 Fazendo previsões no dataset de teste...
🔍 Shape das previsões: (2400, 8)
🔍 Exemplo de previsão (valores): [67.2 35.  67.2 35.  67.2 35.  66.6 35. ]
✅ Previsões salvas em reg_results/predictions_rf_regressor.csv

📈 MÉTRICAS DE REGRESSÃO:
  Coordenada 1: MAE = 17.0248, MSE = 432.5770, R² = -0.7066
  Coordenada 2: MAE = 15.1586, MSE = 347.8124, R² = -1.0593
  Coordenada 3: MAE = 17.0019,

# random igual

In [3]:
#igual ao interpol

In [8]:
import pandas as pd
import numpy as np

def criar_dataset_aleatorio_continuo_sem_restricoes(caminho_entrada, caminho_saida):
    """
    Lê o CSV original e cria uma nova versão com posições finais TOTALMENTE aleatórias
    SEM verificar distância mínima - qualquer posição é válida
    
    Args:
        caminho_entrada: Caminho do CSV original
        caminho_saida: Caminho onde salvar o novo CSV com valores aleatórios
    """
    
    print("📂 Carregando dataset original...")
    df = pd.read_csv(caminho_entrada)
    print(f"✅ Dataset carregado com {len(df)} linhas")
    
    # Valores contínuos para X e Y (área alvo)
    x_min, x_max = 60, 120  # Área alvo X
    y_min, y_max = 0, 60    # Área alvo Y
    
    print("🎲 Gerando valores TOTALMENTE aleatórios (sem restrições de distância)...")
    
    # Gerar valores aleatórios para cada linha
    for index in range(len(df)):
        if index % 1000 == 0:  # Progress indicator
            print(f"Processando linha {index}/{len(df)}")
        
        # Gerar posições TOTALMENTE aleatórias para os 4 UAVs
        for uav in range(1, 5):
            x = np.random.uniform(x_min, x_max)  # ← Qualquer X
            y = np.random.uniform(y_min, y_max)  # ← Qualquer Y
            
            df.loc[index, f'Predicted_Final_X{uav}'] = x
            df.loc[index, f'Predicted_Final_Y{uav}'] = y
    
    # Salvar o novo dataset
    print(f"💾 Salvando dataset aleatório em: {caminho_saida}")
    df.to_csv(caminho_saida, index=False)
    
    print(f"✅ Dataset aleatório criado com sucesso!")
    print(f"📊 {len(df)} linhas processadas")
    
    # Mostrar estatísticas dos valores gerados
    print(f"\n📈 Estatísticas dos valores aleatórios:")
    for uav in range(1, 5):
        x_col = f'Predicted_Final_X{uav}'
        y_col = f'Predicted_Final_Y{uav}'
        
        x_min_val = df[x_col].min()
        x_max_val = df[x_col].max()
        y_min_val = df[y_col].min()
        y_max_val = df[y_col].max()
        
        print(f"  UAV {uav}: X=[{x_min_val:.2f}, {x_max_val:.2f}], Y=[{y_min_val:.2f}, {y_max_val:.2f}]")
    
    # Verificar algumas distâncias (só para informação)
    print(f"\n🔍 Distâncias entre UAVs (primeiras 3 linhas - só informativo):")
    for i in range(min(3, len(df))):
        posicoes = []
        for uav in range(1, 5):
            x = df.loc[i, f'Predicted_Final_X{uav}']
            y = df.loc[i, f'Predicted_Final_Y{uav}']
            posicoes.append((x, y))
        
        # Calcular distâncias (só para mostrar)
        distancias = []
        for j in range(4):
            for k in range(j + 1, 4):
                dist = np.sqrt((posicoes[j][0] - posicoes[k][0])**2 + 
                              (posicoes[j][1] - posicoes[k][1])**2)
                distancias.append(dist)
        
        dist_min = min(distancias)
        print(f"  Linha {i}: Distância mínima = {dist_min:.2f}m")
    
    return df

# ================================
# EXEMPLO DE USO
# ================================

# Definir caminhos
caminho_original = "reg_results/predictions_xgboost_regressor.csv"
caminho_aleatorio = "reg_results/random.csv"

# Criar dataset aleatório SEM restrições
dataset_aleatorio = criar_dataset_aleatorio_continuo_sem_restricoes(caminho_original, caminho_aleatorio)

print(f"\n🎉 PROCESSO CONCLUÍDO!")
print(f"📁 Ficheiro original: {caminho_original}")
print(f"📁 Ficheiro aleatório: {caminho_aleatorio}")


📂 Carregando dataset original...
✅ Dataset carregado com 2400 linhas
🎲 Gerando valores TOTALMENTE aleatórios (sem restrições de distância)...
Processando linha 0/2400
Processando linha 1000/2400
Processando linha 2000/2400
💾 Salvando dataset aleatório em: reg_results/random.csv
✅ Dataset aleatório criado com sucesso!
📊 2400 linhas processadas

📈 Estatísticas dos valores aleatórios:
  UAV 1: X=[60.01, 119.87], Y=[0.00, 60.00]
  UAV 2: X=[60.06, 119.97], Y=[0.02, 59.97]
  UAV 3: X=[60.02, 120.00], Y=[0.03, 60.00]
  UAV 4: X=[60.04, 119.95], Y=[0.05, 59.97]

🔍 Distâncias entre UAVs (primeiras 3 linhas - só informativo):
  Linha 0: Distância mínima = 9.32m
  Linha 1: Distância mínima = 2.92m
  Linha 2: Distância mínima = 19.94m

🎉 PROCESSO CONCLUÍDO!
📁 Ficheiro original: reg_results/predictions_xgboost_regressor.csv
📁 Ficheiro aleatório: reg_results/random.csv


# ANALISAR TUDO

In [4]:
import numpy as np
import networkx as nx
import pandas as pd

def has_collision_final_positions(final_positions, min_distance=20):
    """
    nao verifica
    """
    
    return False  # Sem colisões

def analisar_comunicacoes_interpolado_sem_verificacao_colisoes(initial_positions, final_positions, num_uavs, num_timeslots, jammer_position):
    """
    Analisa comunicações interpolando entre posições inicial e final
    SEM VERIFICAR COLISÕES (para Análise 1)
    """
    
    def criar_best_solution_interpolado(initial_pos, final_pos, num_uavs, num_timeslots):
        """
        Cria best_solution interpolando linearmente entre posições inicial e final
        """
        best_solution = []
        
        for t in range(num_timeslots):
            # Calcular fator de interpolação (0 no início, 1 no final)
            if num_timeslots == 1:
                alpha = 1.0  # Se só há 1 timeslot, usar posição final
            else:
                alpha = t / (num_timeslots - 1)
            
            # Para cada UAV
            for uav in range(num_uavs):
                start_x, start_y = initial_pos[uav]
                end_x, end_y = final_pos[uav]
                
                # Interpolação linear
                x = start_x + alpha * (end_x - start_x)
                y = start_y + alpha * (end_y - start_y)
                
                # Adicionar ao best_solution
                best_solution.extend([x, y])
        
        return best_solution
    
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # CRIAR BEST_SOLUTION ATRAVÉS DE INTERPOLAÇÃO
    best_solution = criar_best_solution_interpolado(initial_positions, final_positions, num_uavs, num_timeslots)
    
    # SEM VERIFICAÇÃO DE COLISÕES - continuar sempre com a análise
    resultados = []
    todos_c_media = []
    todos_c_min = []
    
    # Analisar cada timeslot
    for t in range(num_timeslots):
        positions = []
        angles = []
        
        for i in range(num_uavs):
            idx_pos = t * num_uavs * 2 + i * 2
            x, y = best_solution[idx_pos:idx_pos+2]
            positions.append(np.array([x, y]))
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            angles.append(angle)
        
        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
        
        # Gerar métricas detalhadas
        metricas = avaliar_grafo(comm_matrix)
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Adicionar às listas do epoch
        todos_c_media.append(C_media_total)
        todos_c_min.append(C_min_total)
        
        resultados.append({
            'epoch': 1,
            'timeslot': t+1,
            'metricas': metricas,
            'C_media_total': C_media_total,
            'C_min_total': C_min_total
        })
    
    # Calcular fitness final
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []
    
    for i in range(len(todos_c_media)):
        C_media = todos_c_media[i]
        C_min = todos_c_min[i]
        fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
        fitness_por_timeslot.append(fitness_timeslot)
    
    # Fitness médio do epoch
    fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
    
    # Calcular também as médias para informação
    media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
    media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch

def analisar_comunicacoes_interpolado_final(initial_positions, final_positions, num_uavs, num_timeslots, jammer_position):
    """
    Analisa comunicações interpolando entre posições inicial e final
    VERIFICA COLISÕES APENAS NAS POSIÇÕES FINAIS
    """
    
    def criar_best_solution_interpolado(initial_pos, final_pos, num_uavs, num_timeslots):
        """
        Cria best_solution interpolando linearmente entre posições inicial e final
        """
        best_solution = []
        
        for t in range(num_timeslots):
            # Calcular fator de interpolação (0 no início, 1 no final)
            if num_timeslots == 1:
                alpha = 1.0  # Se só há 1 timeslot, usar posição final
            else:
                alpha = t / (num_timeslots - 1)
            
            # Para cada UAV
            for uav in range(num_uavs):
                start_x, start_y = initial_pos[uav]
                end_x, end_y = final_pos[uav]
                
                # Interpolação linear
                x = start_x + alpha * (end_x - start_x)
                y = start_y + alpha * (end_y - start_y)
                
                # Adicionar ao best_solution
                best_solution.extend([x, y])
        
        return best_solution
    
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # CRIAR BEST_SOLUTION ATRAVÉS DE INTERPOLAÇÃO
    best_solution = criar_best_solution_interpolado(initial_positions, final_positions, num_uavs, num_timeslots)
    
    # 🆕 VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS (não ao longo do caminho)
    tem_colisoes = has_collision_final_positions(final_positions)
    
    if tem_colisoes:
        return [(0.0, "Solução com colisões nas posições finais")], 0.0, 0.0, 0.0
    
    # Se não há colisões nas posições finais, continuar com a análise normal
    resultados = []
    todos_c_media = []
    todos_c_min = []
    
    # Analisar cada timeslot
    for t in range(num_timeslots):
        positions = []
        angles = []
        
        for i in range(num_uavs):
            idx_pos = t * num_uavs * 2 + i * 2
            x, y = best_solution[idx_pos:idx_pos+2]
            positions.append(np.array([x, y]))
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            angles.append(angle)
        
        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
        
        # Gerar métricas detalhadas
        metricas = avaliar_grafo(comm_matrix)
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Adicionar às listas do epoch
        todos_c_media.append(C_media_total)
        todos_c_min.append(C_min_total)
        
        resultados.append({
            'epoch': 1,
            'timeslot': t+1,
            'metricas': metricas,
            'C_media_total': C_media_total,
            'C_min_total': C_min_total
        })
    
    # Calcular fitness final
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []
    
    for i in range(len(todos_c_media)):
        C_media = todos_c_media[i]
        C_min = todos_c_min[i]
        fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
        fitness_por_timeslot.append(fitness_timeslot)
    
    # Fitness médio do epoch
    fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
    
        # Calcular também as médias para informação
    media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
    media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch



In [9]:
def carregar_e_analisar_dataset_final(caminho_dataset, num_uavs=4, num_timeslots=6):
    """
    Carrega o dataset e analisa comunicações para valores reais e preditos
    TRÊS ANÁLISES com verificação de colisões apenas nas posições finais:
    1. Todas as linhas (valores reais)
    2. Apenas sem colisões
    3. Todas as linhas com colisões = 0
    """
    
    # CARREGAR O DATASET
    df = pd.read_csv(caminho_dataset)
    
    # ANÁLISE 1: Todas as linhas (valores reais)
    fitness_reais = []
    fitness_preditos = []
    c_media_reais = []
    c_media_preditos = []
    c_min_reais = []
    c_min_preditos = []
    
    # ANÁLISE 2: Apenas sem colisões
    fitness_reais_sem_colisoes = []
    fitness_preditos_sem_colisoes = []
    c_media_reais_sem_colisoes = []
    c_media_preditos_sem_colisoes = []
    c_min_reais_sem_colisoes = []
    c_min_preditos_sem_colisoes = []
    
    # ANÁLISE 3: Todas as linhas, mas colisões = 0
    fitness_reais_colisoes_zero = []
    fitness_preditos_colisoes_zero = []
    c_media_reais_colisoes_zero = []
    c_media_preditos_colisoes_zero = []
    c_min_reais_colisoes_zero = []
    c_min_preditos_colisoes_zero = []
    
    linhas_sem_colisoes = []
    total_linhas_sem_colisoes = 0
    
    # ANALISAR CADA LINHA DO DATASET
    for index, row in df.iterrows():
        # Extrair posições
        initial_positions = [
            (row['Initial_X1'], row['Initial_Y1']),
            (row['Initial_X2'], row['Initial_Y2']),
            (row['Initial_X3'], row['Initial_Y3']),
            (row['Initial_X4'], row['Initial_Y4'])
        ]
        
        final_positions_preditas = [
            (row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
            (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
            (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
            (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])
        ]
        
        final_positions_reais = [
            (row['Real_Final_X1'], row['Real_Final_Y1']),
            (row['Real_Final_X2'], row['Real_Final_Y2']),
            (row['Real_Final_X3'], row['Real_Final_Y3']),
            (row['Real_Final_X4'], row['Real_Final_Y4'])
        ]
        
        jammer_position = [row['Jammer_X'], row['Jammer_Y']]
        
        # VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS
        tem_colisoes_predito = has_collision_final_positions(final_positions_preditas)
        tem_colisoes_real = has_collision_final_positions(final_positions_reais)
        
        # CALCULAR VALORES REAIS DE COMUNICAÇÃO (mesmo com colisões) - USANDO FUNÇÃO SEM VERIFICAÇÃO
        resultados_pred, fitness_pred, c_media_pred, c_min_pred = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_preditas,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        resultados_real, fitness_real, c_media_real, c_min_real = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_reais,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        # ANÁLISE 1: Armazenar valores reais (independente de colisões)
        fitness_preditos.append(fitness_pred)
        fitness_reais.append(fitness_real)
        c_media_preditos.append(c_media_pred)
        c_media_reais.append(c_media_real)
        c_min_preditos.append(c_min_pred)
        c_min_reais.append(c_min_real)
        
        # ANÁLISE 2: Se não há colisões, adicionar às listas especiais
        if not tem_colisoes_predito:
            fitness_preditos_sem_colisoes.append(fitness_pred)
            fitness_reais_sem_colisoes.append(fitness_real)
            c_media_preditos_sem_colisoes.append(c_media_pred)
            c_media_reais_sem_colisoes.append(c_media_real)
            c_min_preditos_sem_colisoes.append(c_min_pred)
            c_min_reais_sem_colisoes.append(c_min_real)
            
            linhas_sem_colisoes.append(index + 1)
            total_linhas_sem_colisoes += 1
        
        # ANÁLISE 3: Colisões = 0
        # Para preditos
        if tem_colisoes_predito:
            fitness_preditos_colisoes_zero.append(0.0)
            c_media_preditos_colisoes_zero.append(0.0)
            c_min_preditos_colisoes_zero.append(0.0)
        else:
            fitness_preditos_colisoes_zero.append(fitness_pred)
            c_media_preditos_colisoes_zero.append(c_media_pred)
            c_min_preditos_colisoes_zero.append(c_min_pred)
        
        # Para reais
        if tem_colisoes_real:
            fitness_reais_colisoes_zero.append(0.0)
            c_media_reais_colisoes_zero.append(0.0)
            c_min_reais_colisoes_zero.append(0.0)
        else:
            fitness_reais_colisoes_zero.append(fitness_real)
            c_media_reais_colisoes_zero.append(c_media_real)
            c_min_reais_colisoes_zero.append(c_min_real)
    
    # CALCULAR MÉDIAS
    media_fitness_real_1 = np.mean(fitness_reais)
    media_fitness_predito_1 = np.mean(fitness_preditos)
    media_c_media_real_1 = np.mean(c_media_reais)
    media_c_media_predito_1 = np.mean(c_media_preditos)
    media_c_min_real_1 = np.mean(c_min_reais)
    media_c_min_predito_1 = np.mean(c_min_preditos)
    
    media_fitness_real_3 = np.mean(fitness_reais_colisoes_zero)
    media_fitness_predito_3 = np.mean(fitness_preditos_colisoes_zero)
    media_c_media_real_3 = np.mean(c_media_reais_colisoes_zero)
    media_c_media_predito_3 = np.mean(c_media_preditos_colisoes_zero)
    media_c_min_real_3 = np.mean(c_min_reais_colisoes_zero)
    media_c_min_predito_3 = np.mean(c_min_preditos_colisoes_zero)
    
    # ANÁLISE 1: RESULTADOS PARA TODAS AS LINHAS
    print(f"============================================================")
    print(f"ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)")
    print(f"============================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_1:.4f} | Predito={media_fitness_predito_1:.4f} | Diff={abs(media_fitness_real_1 - media_fitness_predito_1):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_1:.2f} | Predito={media_c_media_predito_1:.2f} | Diff={abs(media_c_media_real_1 - media_c_media_predito_1):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_1:.2f} | Predito={media_c_min_predito_1:.2f} | Diff={abs(media_c_min_real_1 - media_c_min_predito_1):.2f}")
    
    # ANÁLISE 2: APENAS SEM COLISÕES
    if total_linhas_sem_colisoes > 0:
        media_fitness_real_2 = np.mean(fitness_reais_sem_colisoes)
        media_fitness_predito_2 = np.mean(fitness_preditos_sem_colisoes)
        media_c_media_real_2 = np.mean(c_media_reais_sem_colisoes)
        media_c_media_predito_2 = np.mean(c_media_preditos_sem_colisoes)
        media_c_min_real_2 = np.mean(c_min_reais_sem_colisoes)
        media_c_min_predito_2 = np.mean(c_min_preditos_sem_colisoes)
        
        print(f"\n======================================================================")
        print(f"ANÁLISE 2 - APENAS LINHAS SEM COLISÕES ({total_linhas_sem_colisoes}/{len(df)} linhas)")
        print(f"======================================================================")
        print(f"🎯 FITNESS: Real={media_fitness_real_2:.4f} | Predito={media_fitness_predito_2:.4f} | Diff={abs(media_fitness_real_2 - media_fitness_predito_2):.4f}")
        print(f"📊 C_MÉDIA: Real={media_c_media_real_2:.2f} | Predito={media_c_media_predito_2:.2f} | Diff={abs(media_c_media_real_2 - media_c_media_predito_2):.2f}")
        print(f"📉 C_MIN: Real={media_c_min_real_2:.2f} | Predito={media_c_min_predito_2:.2f} | Diff={abs(media_c_min_real_2 - media_c_min_predito_2):.2f}")
    
    # ANÁLISE 3: COLISÕES = 0
    print(f"\n======================================================================")
    print(f"ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)")
    print(f"======================================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_3:.4f} | Predito={media_fitness_predito_3:.4f} | Diff={abs(media_fitness_real_3 - media_fitness_predito_3):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_3:.2f} | Predito={media_c_media_predito_3:.2f} | Diff={abs(media_c_media_real_3 - media_c_media_predito_3):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_3:.2f} | Predito={media_c_min_predito_3:.2f} | Diff={abs(media_c_min_real_3 - media_c_min_predito_3):.2f}")

# USO
if __name__ == "__main__":
    caminho_do_dataset = "reg_results/random.csv"
    carregar_e_analisar_dataset_final(caminho_dataset=caminho_do_dataset, num_uavs=4, num_timeslots=6)


ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=328578688.4350 | Predito=15050009.5898 | Diff=313528678.8451
📊 C_MÉDIA: Real=18089.93 | Predito=6630.61 | Diff=11459.32
📉 C_MIN: Real=7415.45 | Predito=1828.05 | Diff=5587.40

ANÁLISE 2 - APENAS LINHAS SEM COLISÕES (2400/2400 linhas)
🎯 FITNESS: Real=328578688.4350 | Predito=15050009.5898 | Diff=313528678.8451
📊 C_MÉDIA: Real=18089.93 | Predito=6630.61 | Diff=11459.32
📉 C_MIN: Real=7415.45 | Predito=1828.05 | Diff=5587.40

ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)
🎯 FITNESS: Real=328578688.4350 | Predito=15050009.5898 | Diff=313528678.8451
📊 C_MÉDIA: Real=18089.93 | Predito=6630.61 | Diff=11459.32
📉 C_MIN: Real=7415.45 | Predito=1828.05 | Diff=5587.40


In [6]:
def carregar_e_analisar_dataset_final(caminho_dataset, num_uavs=4, num_timeslots=6):
    """
    Carrega o dataset e analisa comunicações para valores reais e preditos
    TRÊS ANÁLISES com verificação de colisões apenas nas posições finais:
    1. Todas as linhas (valores reais)
    2. Apenas sem colisões
    3. Todas as linhas com colisões = 0
    """
    
    # CARREGAR O DATASET
    df = pd.read_csv(caminho_dataset)
    
    # ANÁLISE 1: Todas as linhas (valores reais)
    fitness_reais = []
    fitness_preditos = []
    c_media_reais = []
    c_media_preditos = []
    c_min_reais = []
    c_min_preditos = []
    
    # ANÁLISE 2: Apenas sem colisões
    fitness_reais_sem_colisoes = []
    fitness_preditos_sem_colisoes = []
    c_media_reais_sem_colisoes = []
    c_media_preditos_sem_colisoes = []
    c_min_reais_sem_colisoes = []
    c_min_preditos_sem_colisoes = []
    
    # ANÁLISE 3: Todas as linhas, mas colisões = 0
    fitness_reais_colisoes_zero = []
    fitness_preditos_colisoes_zero = []
    c_media_reais_colisoes_zero = []
    c_media_preditos_colisoes_zero = []
    c_min_reais_colisoes_zero = []
    c_min_preditos_colisoes_zero = []
    
    linhas_sem_colisoes = []
    total_linhas_sem_colisoes = 0
    
    # ANALISAR CADA LINHA DO DATASET
    for index, row in df.iterrows():
        # Extrair posições
        initial_positions = [
            (row['Initial_X1'], row['Initial_Y1']),
            (row['Initial_X2'], row['Initial_Y2']),
            (row['Initial_X3'], row['Initial_Y3']),
            (row['Initial_X4'], row['Initial_Y4'])
        ]
        
        final_positions_preditas = [
            (row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
            (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
            (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
            (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])
        ]
        
        final_positions_reais = [
            (row['Real_Final_X1'], row['Real_Final_Y1']),
            (row['Real_Final_X2'], row['Real_Final_Y2']),
            (row['Real_Final_X3'], row['Real_Final_Y3']),
            (row['Real_Final_X4'], row['Real_Final_Y4'])
        ]
        
        jammer_position = [row['Jammer_X'], row['Jammer_Y']]
        
        # VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS
        tem_colisoes_predito = has_collision_final_positions(final_positions_preditas)
        tem_colisoes_real = has_collision_final_positions(final_positions_reais)
        
        # CALCULAR VALORES REAIS DE COMUNICAÇÃO (mesmo com colisões) - USANDO FUNÇÃO SEM VERIFICAÇÃO
        resultados_pred, fitness_pred, c_media_pred, c_min_pred = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_preditas,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        resultados_real, fitness_real, c_media_real, c_min_real = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_reais,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        # ANÁLISE 1: Armazenar valores reais (independente de colisões)
        fitness_preditos.append(fitness_pred)
        fitness_reais.append(fitness_real)
        c_media_preditos.append(c_media_pred)
        c_media_reais.append(c_media_real)
        c_min_preditos.append(c_min_pred)
        c_min_reais.append(c_min_real)
        
        # ANÁLISE 2: Se não há colisões, adicionar às listas especiais
        if not tem_colisoes_predito:
            fitness_preditos_sem_colisoes.append(fitness_pred)
            fitness_reais_sem_colisoes.append(fitness_real)
            c_media_preditos_sem_colisoes.append(c_media_pred)
            c_media_reais_sem_colisoes.append(c_media_real)
            c_min_preditos_sem_colisoes.append(c_min_pred)
            c_min_reais_sem_colisoes.append(c_min_real)
            
            linhas_sem_colisoes.append(index + 1)
            total_linhas_sem_colisoes += 1
        
        # ANÁLISE 3: Colisões = 0
        # Para preditos
        if tem_colisoes_predito:
            fitness_preditos_colisoes_zero.append(0.0)
            c_media_preditos_colisoes_zero.append(0.0)
            c_min_preditos_colisoes_zero.append(0.0)
        else:
            fitness_preditos_colisoes_zero.append(fitness_pred)
            c_media_preditos_colisoes_zero.append(c_media_pred)
            c_min_preditos_colisoes_zero.append(c_min_pred)
        
        # Para reais
        if tem_colisoes_real:
            fitness_reais_colisoes_zero.append(0.0)
            c_media_reais_colisoes_zero.append(0.0)
            c_min_reais_colisoes_zero.append(0.0)
        else:
            fitness_reais_colisoes_zero.append(fitness_real)
            c_media_reais_colisoes_zero.append(c_media_real)
            c_min_reais_colisoes_zero.append(c_min_real)
    
    # CALCULAR MÉDIAS
    media_fitness_real_1 = np.mean(fitness_reais)
    media_fitness_predito_1 = np.mean(fitness_preditos)
    media_c_media_real_1 = np.mean(c_media_reais)
    media_c_media_predito_1 = np.mean(c_media_preditos)
    media_c_min_real_1 = np.mean(c_min_reais)
    media_c_min_predito_1 = np.mean(c_min_preditos)
    
    media_fitness_real_3 = np.mean(fitness_reais_colisoes_zero)
    media_fitness_predito_3 = np.mean(fitness_preditos_colisoes_zero)
    media_c_media_real_3 = np.mean(c_media_reais_colisoes_zero)
    media_c_media_predito_3 = np.mean(c_media_preditos_colisoes_zero)
    media_c_min_real_3 = np.mean(c_min_reais_colisoes_zero)
    media_c_min_predito_3 = np.mean(c_min_preditos_colisoes_zero)
    
    # ANÁLISE 1: RESULTADOS PARA TODAS AS LINHAS
    print(f"============================================================")
    print(f"ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)")
    print(f"============================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_1:.4f} | Predito={media_fitness_predito_1:.4f} | Diff={abs(media_fitness_real_1 - media_fitness_predito_1):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_1:.2f} | Predito={media_c_media_predito_1:.2f} | Diff={abs(media_c_media_real_1 - media_c_media_predito_1):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_1:.2f} | Predito={media_c_min_predito_1:.2f} | Diff={abs(media_c_min_real_1 - media_c_min_predito_1):.2f}")
    
    # ANÁLISE 2: APENAS SEM COLISÕES
    if total_linhas_sem_colisoes > 0:
        media_fitness_real_2 = np.mean(fitness_reais_sem_colisoes)
        media_fitness_predito_2 = np.mean(fitness_preditos_sem_colisoes)
        media_c_media_real_2 = np.mean(c_media_reais_sem_colisoes)
        media_c_media_predito_2 = np.mean(c_media_preditos_sem_colisoes)
        media_c_min_real_2 = np.mean(c_min_reais_sem_colisoes)
        media_c_min_predito_2 = np.mean(c_min_preditos_sem_colisoes)
        
        print(f"\n======================================================================")
        print(f"ANÁLISE 2 - APENAS LINHAS SEM COLISÕES ({total_linhas_sem_colisoes}/{len(df)} linhas)")
        print(f"======================================================================")
        print(f"🎯 FITNESS: Real={media_fitness_real_2:.4f} | Predito={media_fitness_predito_2:.4f} | Diff={abs(media_fitness_real_2 - media_fitness_predito_2):.4f}")
        print(f"📊 C_MÉDIA: Real={media_c_media_real_2:.2f} | Predito={media_c_media_predito_2:.2f} | Diff={abs(media_c_media_real_2 - media_c_media_predito_2):.2f}")
        print(f"📉 C_MIN: Real={media_c_min_real_2:.2f} | Predito={media_c_min_predito_2:.2f} | Diff={abs(media_c_min_real_2 - media_c_min_predito_2):.2f}")
    
    # ANÁLISE 3: COLISÕES = 0
    print(f"\n======================================================================")
    print(f"ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)")
    print(f"======================================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_3:.4f} | Predito={media_fitness_predito_3:.4f} | Diff={abs(media_fitness_real_3 - media_fitness_predito_3):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_3:.2f} | Predito={media_c_media_predito_3:.2f} | Diff={abs(media_c_media_real_3 - media_c_media_predito_3):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_3:.2f} | Predito={media_c_min_predito_3:.2f} | Diff={abs(media_c_min_real_3 - media_c_min_predito_3):.2f}")

# USO
if __name__ == "__main__":
    caminho_do_dataset = "reg_results/predictions_xgboost_regressor.csv"
    carregar_e_analisar_dataset_final(caminho_dataset=caminho_do_dataset, num_uavs=4, num_timeslots=6)


ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=328578688.4350 | Predito=96119149.4831 | Diff=232459538.9518
📊 C_MÉDIA: Real=18089.93 | Predito=14411.51 | Diff=3678.42
📉 C_MIN: Real=7415.45 | Predito=4243.63 | Diff=3171.82

ANÁLISE 2 - APENAS LINHAS SEM COLISÕES (2400/2400 linhas)
🎯 FITNESS: Real=328578688.4350 | Predito=96119149.4831 | Diff=232459538.9518
📊 C_MÉDIA: Real=18089.93 | Predito=14411.51 | Diff=3678.42
📉 C_MIN: Real=7415.45 | Predito=4243.63 | Diff=3171.82

ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)
🎯 FITNESS: Real=328578688.4350 | Predito=96119149.4831 | Diff=232459538.9518
📊 C_MÉDIA: Real=18089.93 | Predito=14411.51 | Diff=3678.42
📉 C_MIN: Real=7415.45 | Predito=4243.63 | Diff=3171.82


In [10]:
def carregar_e_analisar_dataset_final(caminho_dataset, num_uavs=4, num_timeslots=6):
    """
    Carrega o dataset e analisa comunicações para valores reais e preditos
    TRÊS ANÁLISES com verificação de colisões apenas nas posições finais:
    1. Todas as linhas (valores reais)
    2. Apenas sem colisões
    3. Todas as linhas com colisões = 0
    """
    
    # CARREGAR O DATASET
    df = pd.read_csv(caminho_dataset)
    
    # ANÁLISE 1: Todas as linhas (valores reais)
    fitness_reais = []
    fitness_preditos = []
    c_media_reais = []
    c_media_preditos = []
    c_min_reais = []
    c_min_preditos = []
    
    # ANÁLISE 2: Apenas sem colisões
    fitness_reais_sem_colisoes = []
    fitness_preditos_sem_colisoes = []
    c_media_reais_sem_colisoes = []
    c_media_preditos_sem_colisoes = []
    c_min_reais_sem_colisoes = []
    c_min_preditos_sem_colisoes = []
    
    # ANÁLISE 3: Todas as linhas, mas colisões = 0
    fitness_reais_colisoes_zero = []
    fitness_preditos_colisoes_zero = []
    c_media_reais_colisoes_zero = []
    c_media_preditos_colisoes_zero = []
    c_min_reais_colisoes_zero = []
    c_min_preditos_colisoes_zero = []
    
    linhas_sem_colisoes = []
    total_linhas_sem_colisoes = 0
    
    # ANALISAR CADA LINHA DO DATASET
    for index, row in df.iterrows():
        # Extrair posições
        initial_positions = [
            (row['Initial_X1'], row['Initial_Y1']),
            (row['Initial_X2'], row['Initial_Y2']),
            (row['Initial_X3'], row['Initial_Y3']),
            (row['Initial_X4'], row['Initial_Y4'])
        ]
        
        final_positions_preditas = [
            (row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
            (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
            (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
            (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])
        ]
        
        final_positions_reais = [
            (row['Real_Final_X1'], row['Real_Final_Y1']),
            (row['Real_Final_X2'], row['Real_Final_Y2']),
            (row['Real_Final_X3'], row['Real_Final_Y3']),
            (row['Real_Final_X4'], row['Real_Final_Y4'])
        ]
        
        jammer_position = [row['Jammer_X'], row['Jammer_Y']]
        
        # VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS
        tem_colisoes_predito = has_collision_final_positions(final_positions_preditas)
        tem_colisoes_real = has_collision_final_positions(final_positions_reais)
        
        # CALCULAR VALORES REAIS DE COMUNICAÇÃO (mesmo com colisões) - USANDO FUNÇÃO SEM VERIFICAÇÃO
        resultados_pred, fitness_pred, c_media_pred, c_min_pred = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_preditas,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        resultados_real, fitness_real, c_media_real, c_min_real = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_reais,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        # ANÁLISE 1: Armazenar valores reais (independente de colisões)
        fitness_preditos.append(fitness_pred)
        fitness_reais.append(fitness_real)
        c_media_preditos.append(c_media_pred)
        c_media_reais.append(c_media_real)
        c_min_preditos.append(c_min_pred)
        c_min_reais.append(c_min_real)
        
        # ANÁLISE 2: Se não há colisões, adicionar às listas especiais
        if not tem_colisoes_predito:
            fitness_preditos_sem_colisoes.append(fitness_pred)
            fitness_reais_sem_colisoes.append(fitness_real)
            c_media_preditos_sem_colisoes.append(c_media_pred)
            c_media_reais_sem_colisoes.append(c_media_real)
            c_min_preditos_sem_colisoes.append(c_min_pred)
            c_min_reais_sem_colisoes.append(c_min_real)
            
            linhas_sem_colisoes.append(index + 1)
            total_linhas_sem_colisoes += 1
        
        # ANÁLISE 3: Colisões = 0
        # Para preditos
        if tem_colisoes_predito:
            fitness_preditos_colisoes_zero.append(0.0)
            c_media_preditos_colisoes_zero.append(0.0)
            c_min_preditos_colisoes_zero.append(0.0)
        else:
            fitness_preditos_colisoes_zero.append(fitness_pred)
            c_media_preditos_colisoes_zero.append(c_media_pred)
            c_min_preditos_colisoes_zero.append(c_min_pred)
        
        # Para reais
        if tem_colisoes_real:
            fitness_reais_colisoes_zero.append(0.0)
            c_media_reais_colisoes_zero.append(0.0)
            c_min_reais_colisoes_zero.append(0.0)
        else:
            fitness_reais_colisoes_zero.append(fitness_real)
            c_media_reais_colisoes_zero.append(c_media_real)
            c_min_reais_colisoes_zero.append(c_min_real)
    
    # CALCULAR MÉDIAS
    media_fitness_real_1 = np.mean(fitness_reais)
    media_fitness_predito_1 = np.mean(fitness_preditos)
    media_c_media_real_1 = np.mean(c_media_reais)
    media_c_media_predito_1 = np.mean(c_media_preditos)
    media_c_min_real_1 = np.mean(c_min_reais)
    media_c_min_predito_1 = np.mean(c_min_preditos)
    
    media_fitness_real_3 = np.mean(fitness_reais_colisoes_zero)
    media_fitness_predito_3 = np.mean(fitness_preditos_colisoes_zero)
    media_c_media_real_3 = np.mean(c_media_reais_colisoes_zero)
    media_c_media_predito_3 = np.mean(c_media_preditos_colisoes_zero)
    media_c_min_real_3 = np.mean(c_min_reais_colisoes_zero)
    media_c_min_predito_3 = np.mean(c_min_preditos_colisoes_zero)
    
    # ANÁLISE 1: RESULTADOS PARA TODAS AS LINHAS
    print(f"============================================================")
    print(f"ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)")
    print(f"============================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_1:.4f} | Predito={media_fitness_predito_1:.4f} | Diff={abs(media_fitness_real_1 - media_fitness_predito_1):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_1:.2f} | Predito={media_c_media_predito_1:.2f} | Diff={abs(media_c_media_real_1 - media_c_media_predito_1):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_1:.2f} | Predito={media_c_min_predito_1:.2f} | Diff={abs(media_c_min_real_1 - media_c_min_predito_1):.2f}")
    
    # ANÁLISE 2: APENAS SEM COLISÕES
    if total_linhas_sem_colisoes > 0:
        media_fitness_real_2 = np.mean(fitness_reais_sem_colisoes)
        media_fitness_predito_2 = np.mean(fitness_preditos_sem_colisoes)
        media_c_media_real_2 = np.mean(c_media_reais_sem_colisoes)
        media_c_media_predito_2 = np.mean(c_media_preditos_sem_colisoes)
        media_c_min_real_2 = np.mean(c_min_reais_sem_colisoes)
        media_c_min_predito_2 = np.mean(c_min_preditos_sem_colisoes)
        
        print(f"\n======================================================================")
        print(f"ANÁLISE 2 - APENAS LINHAS SEM COLISÕES ({total_linhas_sem_colisoes}/{len(df)} linhas)")
        print(f"======================================================================")
        print(f"🎯 FITNESS: Real={media_fitness_real_2:.4f} | Predito={media_fitness_predito_2:.4f} | Diff={abs(media_fitness_real_2 - media_fitness_predito_2):.4f}")
        print(f"📊 C_MÉDIA: Real={media_c_media_real_2:.2f} | Predito={media_c_media_predito_2:.2f} | Diff={abs(media_c_media_real_2 - media_c_media_predito_2):.2f}")
        print(f"📉 C_MIN: Real={media_c_min_real_2:.2f} | Predito={media_c_min_predito_2:.2f} | Diff={abs(media_c_min_real_2 - media_c_min_predito_2):.2f}")
    
    # ANÁLISE 3: COLISÕES = 0
    print(f"\n======================================================================")
    print(f"ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)")
    print(f"======================================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_3:.4f} | Predito={media_fitness_predito_3:.4f} | Diff={abs(media_fitness_real_3 - media_fitness_predito_3):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_3:.2f} | Predito={media_c_media_predito_3:.2f} | Diff={abs(media_c_media_real_3 - media_c_media_predito_3):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_3:.2f} | Predito={media_c_min_predito_3:.2f} | Diff={abs(media_c_min_real_3 - media_c_min_predito_3):.2f}")

# USO
if __name__ == "__main__":
    caminho_do_dataset = "reg_results/predictions_rf_regressor.csv"
    carregar_e_analisar_dataset_final(caminho_dataset=caminho_do_dataset, num_uavs=4, num_timeslots=6)


ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=328578688.4350 | Predito=733294250418699.2500 | Diff=733293921840010.8750
📊 C_MÉDIA: Real=18089.93 | Predito=6295087.69 | Diff=6276997.76
📉 C_MIN: Real=7415.45 | Predito=3030124.39 | Diff=3022708.94

ANÁLISE 2 - APENAS LINHAS SEM COLISÕES (2400/2400 linhas)
🎯 FITNESS: Real=328578688.4350 | Predito=733294250418699.2500 | Diff=733293921840010.8750
📊 C_MÉDIA: Real=18089.93 | Predito=6295087.69 | Diff=6276997.76
📉 C_MIN: Real=7415.45 | Predito=3030124.39 | Diff=3022708.94

ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)
🎯 FITNESS: Real=328578688.4350 | Predito=733294250418699.2500 | Diff=733293921840010.8750
📊 C_MÉDIA: Real=18089.93 | Predito=6295087.69 | Diff=6276997.76
📉 C_MIN: Real=7415.45 | Predito=3030124.39 | Diff=3022708.94


# verifica colisoes

In [5]:
import pandas as pd
import numpy as np

def verificar_colisoes(df, distancia_minima=20):
    """Verifica colisões nas posições iniciais, previstas e reais finais para cada linha."""
    
    # Função para calcular a distância entre dois pontos
    def calcular_distancia(p1, p2):
        return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
    
    # Armazenar resultados de colisões
    resultados = []
    
    # Iterar sobre cada linha do DataFrame
    for index, row in df.iterrows():
        colisoes = {
            'linha': index,
            'colisoes_iniciais': [],
            'colisoes_previsoes': [],
            'colisoes_reais': []
        }
        
        # Verificar colisões nas posições iniciais
        iniciais = [(row['Initial_X1'], row['Initial_Y1']),
                    (row['Initial_X2'], row['Initial_Y2']),
                    (row['Initial_X3'], row['Initial_Y3']),
                    (row['Initial_X4'], row['Initial_Y4'])]
        
        for i in range(len(iniciais)):
            for j in range(i + 1, len(iniciais)):
                if calcular_distancia(iniciais[i], iniciais[j]) < distancia_minima:
                    colisoes['colisoes_iniciais'].append((i + 1, j + 1))  # +1 para indicar UAVs 1 a 4
        
        # Verificar colisões nas previsões
        previsoes = [(row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
                     (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
                     (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
                     (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])]
        
        for i in range(len(previsoes)):
            for j in range(i + 1, len(previsoes)):
                if calcular_distancia(previsoes[i], previsoes[j]) < distancia_minima:
                    colisoes['colisoes_previsoes'].append((i + 1, j + 1))  # +1 para indicar UAVs 1 a 4
        
        # Verificar colisões nas posições reais finais
        reais = [(row['Real_Final_X1'], row['Real_Final_Y1']),
                 (row['Real_Final_X2'], row['Real_Final_Y2']),
                 (row['Real_Final_X3'], row['Real_Final_Y3']),
                 (row['Real_Final_X4'], row['Real_Final_Y4'])]
        
        for i in range(len(reais)):
            for j in range(i + 1, len(reais)):
                if calcular_distancia(reais[i], reais[j]) < distancia_minima:
                    colisoes['colisoes_reais'].append((i + 1, j + 1))  # +1 para indicar UAVs 1 a 4
        
        resultados.append(colisoes)
    
    return resultados

# ================================
# USO
# ================================

# Carregar o dataset com previsões
file_path = 'reg_results/predictions_rf_regressor.csv'  # Caminho do arquivo com previsões
df = pd.read_csv(file_path)

# Verificar colisões
resultados_colisoes = verificar_colisoes(df)

# Exibir resultados
for resultado in resultados_colisoes:
    print(f"Linha {resultado['linha']}:")
    print(f"  Colisões nas posições iniciais: {resultado['colisoes_iniciais']}")
    print(f"  Colisões nas previsões: {resultado['colisoes_previsoes']}")
    print(f"  Colisões nas posições reais finais: {resultado['colisoes_reais']}")


Linha 0:
  Colisões nas posições iniciais: []
  Colisões nas previsões: [(1, 2), (2, 4)]
  Colisões nas posições reais finais: []
Linha 1:
  Colisões nas posições iniciais: []
  Colisões nas previsões: [(1, 4)]
  Colisões nas posições reais finais: []
Linha 2:
  Colisões nas posições iniciais: []
  Colisões nas previsões: [(2, 3)]
  Colisões nas posições reais finais: []
Linha 3:
  Colisões nas posições iniciais: []
  Colisões nas previsões: []
  Colisões nas posições reais finais: []
Linha 4:
  Colisões nas posições iniciais: []
  Colisões nas previsões: []
  Colisões nas posições reais finais: []
Linha 5:
  Colisões nas posições iniciais: []
  Colisões nas previsões: [(1, 3)]
  Colisões nas posições reais finais: []
Linha 6:
  Colisões nas posições iniciais: []
  Colisões nas previsões: []
  Colisões nas posições reais finais: []
Linha 7:
  Colisões nas posições iniciais: []
  Colisões nas previsões: []
  Colisões nas posições reais finais: []
Linha 8:
  Colisões nas posições iniciai

# MED

In [11]:
import pandas as pd
import numpy as np
import os
import glob

def calculate_mean_euclidean_distance_single(dataset_file):
    """
    Calcula a Mean Euclidean Distance para um único ficheiro
    
    Args:
        dataset_file: Caminho do ficheiro CSV
    
    Returns:
        dict: Estatísticas da distância euclidiana
    """
    
    try:
        # Carregar dataset
        df = pd.read_csv(dataset_file)
        
        # Lista para armazenar todas as distâncias
        all_distances = []
        
        # Calcular distância euclidiana para cada linha e cada UAV
        for index, row in df.iterrows():
            for uav in range(1, 5):  # UAVs 1, 2, 3, 4
                # Posições preditas
                pred_x = row[f'Predicted_Final_X{uav}']
                pred_y = row[f'Predicted_Final_Y{uav}']
                
                # Posições reais
                real_x = row[f'Real_Final_X{uav}']
                real_y = row[f'Real_Final_Y{uav}']
                
                # Calcular distância euclidiana
                distance = np.sqrt((pred_x - real_x)**2 + (pred_y - real_y)**2)
                all_distances.append(distance)
        
        # Calcular estatísticas
        results = {
            'filename': os.path.basename(dataset_file),
            'total_predictions': len(all_distances),
            'total_lines': len(df),
            'mean_distance': np.mean(all_distances),
            'std_distance': np.std(all_distances),
            'min_distance': np.min(all_distances),
            'max_distance': np.max(all_distances),
            'median_distance': np.median(all_distances)
        }
        
        return results
        
    except Exception as e:
        print(f"❌ Erro ao processar {dataset_file}: {e}")
        return None

def analyze_multiple_files(file_list=None, directory=None, pattern="*.csv"):
    """
    Analisa múltiplos ficheiros e calcula Mean Euclidean Distance
    
    Args:
        file_list: Lista de caminhos de ficheiros (opcional)
        directory: Diretório para procurar ficheiros (opcional)
        pattern: Padrão de ficheiros a procurar (default: "*.csv")
    """
    
    # Determinar lista de ficheiros
    if file_list:
        files = file_list
    elif directory:
        files = glob.glob(os.path.join(directory, pattern))
    else:
        print("❌ Deve fornecer file_list ou directory")
        return
    
    if not files:
        print("❌ Nenhum ficheiro encontrado")
        return
    
    print(f"🔍 Encontrados {len(files)} ficheiros para analisar")
    print("="*80)
    
    # Analisar cada ficheiro
    results = []
    
    for file_path in files:
        print(f"📊 Analisando: {os.path.basename(file_path)}")
        result = calculate_mean_euclidean_distance_single(file_path)
        
        if result:
            results.append(result)
            print(f"   ✅ Mean Euclidean Distance: {result['mean_distance']:.4f}")
        else:
            print(f"   ❌ Falhou")
        print()
    
    # Mostrar resumo comparativo
    if results:
        print("="*80)
        print("📈 RESUMO COMPARATIVO - MEAN EUCLIDEAN DISTANCE")
        print("="*80)
        
        # Cabeçalho
        print(f"{'Ficheiro':<40} {'Mean Dist':<12} {'Std':<10} {'Min':<10} {'Max':<10} {'Linhas':<8}")
        print("-" * 90)
        
        # Resultados
        for result in results:
            filename = result['filename'][:37] + "..." if len(result['filename']) > 40 else result['filename']
            print(f"{filename:<40} {result['mean_distance']:<12.4f} {result['std_distance']:<10.4f} "
                  f"{result['min_distance']:<10.4f} {result['max_distance']:<10.4f} {result['total_lines']:<8}")
        
        # Encontrar melhor e pior
        best = min(results, key=lambda x: x['mean_distance'])
        worst = max(results, key=lambda x: x['mean_distance'])
        
        print("\n" + "="*50)
        print("🏆 RANKING:")
        print(f"   🥇 MELHOR: {best['filename']} (Mean: {best['mean_distance']:.4f})")
        print(f"   🥉 PIOR:   {worst['filename']} (Mean: {worst['mean_distance']:.4f})")
        print(f"   📊 DIFERENÇA: {worst['mean_distance'] - best['mean_distance']:.4f}")
        
        return results
    
    else:
        print("❌ Nenhum ficheiro foi processado com sucesso")
        return None

# Função de conveniência para usar facilmente
def quick_analysis(files):
    """
    Análise rápida de uma lista de ficheiros
    
    Args:
        files: Lista de caminhos de ficheiros
    """
    return analyze_multiple_files(file_list=files)

# Executar
if __name__ == "__main__":
    
    # OPÇÃO 1: Lista específica de ficheiros
    files_to_analyze = [
        "reg_results/predictions_rf_regressor.csv",
        "reg_results/predictions_xgboost_regressor.csv",
        "reg_results/random.csv"
    ]
    
    print("🚀 ANÁLISE DE MEAN EUCLIDEAN DISTANCE")
    print("="*80)
    
    results = quick_analysis(files_to_analyze)
    
    # OPÇÃO 2: Todos os ficheiros de um diretório
    # results = analyze_multiple_files(directory="class_results", pattern="*.csv")
    
    # OPÇÃO 3: Ficheiro individual
    # result = calculate_mean_euclidean_distance_single("inter_results/1n_knn.csv")
    # print(f"Mean Distance: {result['mean_distance']:.4f}")


🚀 ANÁLISE DE MEAN EUCLIDEAN DISTANCE
🔍 Encontrados 3 ficheiros para analisar
📊 Analisando: predictions_rf_regressor.csv
   ✅ Mean Euclidean Distance: 25.0941

📊 Analisando: predictions_xgboost_regressor.csv
   ✅ Mean Euclidean Distance: 18.9362

📊 Analisando: random.csv
   ✅ Mean Euclidean Distance: 29.0346

📈 RESUMO COMPARATIVO - MEAN EUCLIDEAN DISTANCE
Ficheiro                                 Mean Dist    Std        Min        Max        Linhas  
------------------------------------------------------------------------------------------
predictions_rf_regressor.csv             25.0941      12.2338    0.0794     63.3146    2400    
predictions_xgboost_regressor.csv        18.9362      8.8445     0.4321     43.2789    2400    
random.csv                               29.0346      13.8291    0.2287     77.7446    2400    

🏆 RANKING:
   🥇 MELHOR: predictions_xgboost_regressor.csv (Mean: 18.9362)
   🥉 PIOR:   random.csv (Mean: 29.0346)
   📊 DIFERENÇA: 10.0984
